## 1. Download the data

In [ ]:
!gdown https://drive.google.com/uc?id=1Rd5QS0PyKoq9XrIBrg3eLV-ST3tRXjYQ

In [ ]:
# Unzip the downloaded file 
!mkdir -p unzipped_data
!unzip -q data_sans_canonical.zip -d unzipped_data

In [ ]:
# move the unzipped data to the current directory and remove the unzipped directory
!mv unzipped_data/new_data .
!rm -rf unzipped_data

In [ ]:
# Delete the downloaded zip file to make space
!rm -rf data_sans_canonical.zip

In [ ]:
# Remove the existing train, test, and valid directories if they exist
!rm -rf train test valid

## 2. Split the dataset into train, test and validation folders

In [ ]:
import os
import random
import shutil

def copy_images(image_list, target_dir, record_list):
    """
    Copies images to the specified target directory, preserving their class subdirectories.

    Args:
        image_list (list): A list of tuples (subdir, image_path).
        target_dir (str): The base directory to copy images into.
        record_list (list): A list to record copied image metadata (class and path).
    """

    for subdir, src_path in image_list:
        dest_subdir = os.path.join(target_dir, subdir)
        
        # Define destination path
        dest_path = os.path.join(dest_subdir, os.path.basename(src_path))
        os.makedirs(dest_subdir, exist_ok=True)
        try:
            # copy the image
            shutil.copy(src_path, dest_path)
        except FileNotFoundError:
            print(f"Missing file: {src_path}")
        record_list.append((subdir, dest_path))

def train_test_split(root_dir, train_dir, test_dir, valid_dir, split_ratio=0.8):
    """
    Splits images from the root directory into train, test, and validation sets.

    Assumes a directory structure where each subdirectory in `root_dir` corresponds to a class label
    and contains image files.

    Args:
        root_dir (str): Path to the root image dataset directory.
        train_dir (str): Output directory for the training set.
        test_dir (str): Output directory for the testing set.
        valid_dir (str): Output directory for the validation set.
        split_ratio (float): Proportion of data to use for training (default is 0.8).

    Returns:
        tuple: Lists of (class, image_path) tuples for train, test, and validation sets.
    """
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)
    os.makedirs(valid_dir, exist_ok=True)

    train_list, test_list, valid_list = [], [], []

    # Iterate over each class folder in the root directory
    for subdir in os.listdir(root_dir):
        subdir_path = os.path.join(root_dir, subdir)
        
        # Skip if it's not a directory
        if not os.path.isdir(subdir_path):
            continue
        
        # Get all valid image files in the subdirectory
        images = [
            os.path.join(subdir_path, img)
            for img in os.listdir(subdir_path)
            if img.lower().endswith(('.jpg', '.png', '.jpeg')) and os.path.isfile(os.path.join(subdir_path, img))
        ]
        
        # Skip classes with no valid images
        if not images:
            continue
        
        # Shuffle images for random splitting
        random.shuffle(images)

        # Determine split counts
        num_images = len(images)
        num_train = int(num_images * split_ratio)
        num_test = (num_images - num_train) // 2

        # Prepare the split image lists
        train_images = [(subdir, p) for p in images[:num_train]]
        test_images  = [(subdir, p) for p in images[num_train:num_train + num_test]]
        valid_images = [(subdir, p) for p in images[num_train + num_test:]]

        # Copy images to their respective directories 
        copy_images(train_images, train_dir, train_list)
        copy_images(test_images, test_dir, test_list)
        copy_images(valid_images, valid_dir, valid_list)

    total = len(train_list) + len(test_list) + len(valid_list)
    # Print summary
    print(f"Split {total} images into:")
    print(f"  Train: {len(train_list)}")
    print(f"  Test: {len(test_list)}")
    print(f"  Valid: {len(valid_list)}")

    return train_list, test_list, valid_list


In [ ]:
train_list, test_list, valid_list = train_test_split('new_data', 'train', 'test', 'valid')

In [ ]:
len(os.listdir('valid'))

## 3. Create the codepoint to index mapping for labelling purpose

In [ ]:
import requests

url = "https://www.unicode.org/Public/UCD/latest/ucd/UnicodeData.txt"
response = requests.get(url)

# Save to file
with open("UnicodeData.txt", "w", encoding="utf-8") as f:
    f.write(response.text)
 
print("Downloaded UnicodeData.txt")

url = "https://www.unicode.org/Public/UCD/latest/ucd/Blocks.txt"
response = requests.get(url)

# Save to file
with open("Blocks.txt", "w", encoding="utf-8") as f:
    f.write(response.text)

print("Downloaded Blocks.txt")

In [1]:
alphabets = {}

with open('Blocks.txt', 'r') as infile:
  for line in infile.readlines():
    # Skip the comments and new line
    if line[0]=='#' or line[0]=='\n':
      continue

    # Extract the codepoints and the script name
    parts = line.split(";")
    codepoints = parts[0]
    script = parts[1].strip()

    # Separate the starting and ending codepoints
    start_codepoint = codepoints.split('..')[0]
    end_codepoint = codepoints.split('..')[1]

    if any(key in script.lower() for key in ('latin', 'cyrillic', 'armenian',
                                             'greek', 'coptic', 'ipa',
                                             'spacing', 'diacritical','georgian',
                                             'hangul','ethiopic', 'cherokee',
                                             'canadian', 'ogham', 'runic',
                                             'tagalog','hanunoo','buhid',
                                             'tagbanwa')):
      if 'cjk' not in script.lower():
        alphabets[script] = {
            'start': start_codepoint,
            'end': end_codepoint
        }

# Sort the dictionary
alphabets = dict(sorted(alphabets.items()))

In [2]:
import os

# Define acceptable general categories (from Unicode standard)
renderable_categories = {
    'Lu', 'Ll', 'Lt', 'Lm', 'Lo',  # Letters
    'Mn', 'Mc', 'Me',              # Marks
    'Nd', 'Nl', 'No',              # Numbers
    'Pc', 'Pd', 'Ps', 'Pe', 'Pi', 'Pf', 'Po',  # Punctuation
    'Sm', 'Sc', 'Sk', 'So',        # Symbols
    # 'Zs'                         # Space separator (excluded)
}

codepoint_to_idx = {}
count = 0

# Load Unicode data once
with open('UnicodeData.txt', 'r') as infile:
    unicode_lines = infile.readlines()

# Iterate over script ranges
for script, range_info in alphabets.items():
    start_int = int(range_info['start'], 16)
    end_int = int(range_info['end'], 16)

    for line in unicode_lines:
        parts = line.strip().split(';')
        if len(parts) < 3:
            continue
            
        codepoint, name, category,_, _, decomposition = parts[0], parts[1], parts[2], parts[3], parts[4], parts[5]
        
        # Skip if canonical decomposition exists
        if decomposition and not decomposition.startswith('<'):
            continue

        cp_int = int(codepoint, 16)

        if start_int <= cp_int <= end_int:
            if category in renderable_categories:
                codepoint_to_idx['U+'+codepoint] = count
                count += 1


In [ ]:
# Check for duplicate values
value_counts = {}
for key, value in codepoint_to_idx.items():
    if value in value_counts:
        print(f"Duplicate value {value} found for keys: {value_counts[value]} and {key}")
    else:
        value_counts[value] = key

# Alternatively, check using set length
if len(set(codepoint_to_idx.values())) != len(codepoint_to_idx):
    print("Duplicate values found!")
else:
    print("All values are unique.")


In [4]:
idx_to_codepoint = {v: k for k, v in codepoint_to_idx.items()}

In [ ]:
'U+0021' in codepoint_to_idx.keys()

## 4. Create the ground truth

In [ ]:
!wget -O confusables_unicode.txt https://raw.githubusercontent.com/unicode-org/unicodetools/main/unicodetools/data/security/dev/confusables.txt

In [6]:
from collections import defaultdict

class DisjointSet:
    def __init__(self):
        self.rank=defaultdict(int)
        self.parent=defaultdict(str)

    def find(self, node):
        if node not in self.parent:
            self.parent[node] = node

        if self.parent[node]==node:
            return node
        self.parent[node]=self.find(self.parent[node]) # Path compression
        return self.parent[node]

    def union(self, u, v):
        ulp_u = self.find(u)
        ulp_v = self.find(v)

        if ulp_u==ulp_v:
            return
            
        if self.rank[ulp_u]<self.rank[ulp_v]:
            self.parent[ulp_u] = ulp_v
        elif self.rank[ulp_v] < self.rank[ulp_u]:
            self.parent[ulp_v] = ulp_u
        else:
            self.parent[ulp_v] = ulp_u
            self.rank[ulp_u]+=1

In [ ]:
ds = DisjointSet()

with open('confusables_unicode.txt', 'r') as infile:
    for line in infile:
        line = line.strip()
        if not line or line.startswith('#'):
            continue

        parts = line.split(';')
        if len(parts)<2:
            continue

        target_parts = parts[1].strip().split(' ')
        if len(target_parts)>1:
            continue

        src = 'U+' + parts[0].strip()
        dest = 'U+' + parts[1].strip()
        ds.union(src,dest)

# Step 2: Build clusters (transitive groups)
clusters = defaultdict(set)
for cp in ds.parent:
    root = ds.find(cp)
    clusters[root].add(cp)

# Step 3: Keep only codepoints from the LTR scripts
codepoints = set(codepoint_to_idx.keys())

# Make a fully symmetric confusables map
symmetric_confusables = {}

for group in clusters.values():
    group = list(group & codepoints)  # Keep only LTR codepoints
    for cp in group:
        symmetric_confusables[cp] = sorted([other for other in group if other != cp])

In [ ]:
symmetric_confusables['U+0041'], symmetric_confusables['U+0391'], symmetric_confusables['U+0410'], symmetric_confusables['U+13AA']

In [ ]:
symmetric_confusables['U+0026']

## 6. Prepare the dataset for training

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import random
import tensorflow as tf
from pathlib import Path
from keras import applications
from keras import layers
from keras import losses
from keras import ops
from keras import optimizers
from keras import metrics
from keras import Model
from keras.applications import resnet
# Convert keys and values to TensorFlow tensors for lookup
codepoint_keys = tf.constant(list(codepoint_to_idx.keys()))
codepoint_values = tf.constant(list(codepoint_to_idx.values()), dtype=tf.int64)

# Create a lookup table
table = tf.lookup.StaticHashTable(
    initializer=tf.lookup.KeyValueTensorInitializer(codepoint_keys, codepoint_values),
    default_value=-1  # use -1 if codepoint not found
)

augment = tf.keras.Sequential([
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1, 0.1),
    layers.RandomContrast(0.1)
])

def preprocess_image(image_path, folder_name, image_size, training=False, rescale=True):
    image_encoded = tf.io.read_file(image_path)
    image_decoded = tf.image.decode_png(image_encoded, channels=3)
    image_resized = tf.image.resize(image_decoded, image_size)

    if rescale:
        image_resized = image_resized / 255.0 
    
    if training:
        image_resized = augment(image_resized, training=True)
        
    label = table.lookup(folder_name)
    label = tf.cast(label, tf.int32)
    
    return (image_resized, label)

In [11]:
def sample_k_images_for_charid(cid, cids, fids, batch_k):
  possible_fids = tf.boolean_mask(fids, tf.equal(cids, cid))
  count = tf.shape(possible_fids)[0]
  padded_count = tf.cast(tf.math.ceil(batch_k / tf.cast(count, tf.float32)), tf.int32) * count
  full_range = tf.math.mod(tf.range(padded_count), count)

  shuffled = tf.random.shuffle(full_range)
  selected_fids = tf.gather(possible_fids, shuffled[:batch_k])

  return selected_fids, tf.fill([batch_k], cid)

def create_dataset(root_dir, k=4, batch_size=32, image_size=(64,64), training=True, rescale=True):
  folders = os.listdir(root_dir)
  filenames = []

  for folder in folders:
      folder_path = os.path.join(root_dir, folder)
      for file in os.listdir(folder_path):
          full_path = os.path.join(folder_path, file)
          filenames.append(full_path)

  cids = []
  fids = []

  for filename in filenames:
    basename = os.path.basename(filename)
    label = basename.split('_')[0]  # e.g., 'U+0041'
    cids.append(label)
    fids.append(filename)
  
  unique_cids = np.unique(cids)
  dataset = tf.data.Dataset.from_tensor_slices(unique_cids)
  dataset = dataset.shuffle(len(unique_cids))

  batch_k = k
  batch_p = batch_size // k
    
  dataset = dataset.take((len(unique_cids) // batch_p) * batch_p)
  dataset = dataset.repeat(None)  
  print(len(unique_cids))

  dataset = dataset.map(lambda cid: sample_k_images_for_charid(
        cid, cids=cids, fids=fids, batch_k=batch_k))
  
  dataset = dataset.unbatch()

  dataset = dataset.map(
        lambda image_path, folder_name: preprocess_image(
            image_path, folder_name,
            image_size=image_size, training=training, rescale=rescale),
        num_parallel_calls=tf.data.AUTOTUNE)
  
  dataset = dataset.batch(batch_size)
  dataset = dataset.prefetch(1)

  return dataset

In [12]:
def create_eval_dataset(root_dir, image_size=(64, 64), batch_size=32, rescale=True, training=False):
    image_paths = []
    labels = []

    folder_names = os.listdir(root_dir)
    random.shuffle(folder_names)

    for folder in folder_names:
        folder_path = os.path.join(root_dir, folder)
        for fname in os.listdir(folder_path):
            full_path = os.path.join(folder_path, fname)
            image_paths.append(full_path)
            labels.append(folder)  # folder name is label (codepoint)

    image_ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))

    image_ds = image_ds.map(
        lambda path, folder: preprocess_image(path, folder, image_size, rescale=rescale, training=training),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    image_ds = image_ds.batch(batch_size)
    image_ds = image_ds.prefetch(tf.data.AUTOTUNE)
    return image_ds

In [ ]:
train_dataset=create_dataset('train', image_size=(64,64), rescale=False, batch_size=64)

In [15]:
test_dataset = create_eval_dataset('test', image_size=(64,64), rescale=False, batch_size=64)

In [16]:
valid_dataset = create_eval_dataset('valid', image_size=(64,64), rescale=False,batch_size=64)

In [ ]:
import matplotlib.pyplot as plt

for image_batch, label_batch in train_dataset.take(1):
    print("Image batch shape:", image_batch.shape)
    print("Label batch shape:", label_batch.shape)
    
    for i in range(len(image_batch)):  # show up to 8 images
      plt.imshow(image_batch[i].numpy())
      plt.title(f"Label: {idx_to_codepoint[label_batch[i].numpy()]}")
      plt.axis("off")
      plt.show()

In [ ]:
train_dataset.cardinality().numpy(),test_dataset.cardinality().numpy(),valid_dataset.cardinality().numpy()

## 7. Train the model

In [49]:
!pip install keras-tuner -q

In [20]:
import matplotlib.pyplot as plt
import numpy as np
import os
import random
import tensorflow as tf
from pathlib import Path
from keras import applications
from keras import layers
from keras import losses
from keras import ops
from keras import optimizers
from keras import metrics
from keras import Model
from keras.applications import resnet
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.callbacks import ModelCheckpoint
# import keras_tuner
from loss.tripletloss import TripletSemiHardLoss

In [21]:
import tensorflow as tf
from tensorflow.keras import backend as K

class L2_SP(tf.keras.regularizers.Regularizer):
    def __init__(self, base_weights, l2=0.01):
        if l2 is None or l2 < 0:
            raise ValueError("l2 must be a non-negative float.")
        self.l2 = K.cast_to_floatx(l2)
        self.base_weights_tensor = tf.convert_to_tensor(base_weights, dtype=tf.float32)

    def __call__(self, x):
        # Equivalent to l2 * ||x - base||^2
        return self.l2 * tf.reduce_sum(tf.square(x - self.base_weights_tensor))

    def get_config(self):
        return {"l2": float(self.l2)}  # Do NOT include tensors or arrays



In [22]:
def get_embedding_model(input_shape=(64,64,3), backbone='EfficientNetB0', embedding_dim=256, layer_to_freeze='conv5'):
    inputs = layers.Input(shape=input_shape)
    backbone_class = getattr(tf.keras.applications, backbone)
    base_cnn = backbone_class(include_top=False, input_tensor=inputs, weights="imagenet")

    freeze=False

    for layer in base_cnn.layers:
        if layer.name.startswith(layer_to_freeze):
            freeze=True
        layer.trainable=freeze

    x = layers.Flatten()(base_cnn.output)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(embedding_dim, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(embedding_dim)(x)
    output = layers.Lambda(lambda x: tf.math.l2_normalize(x, axis=1), 
                           name="l2_normalized", 
                           output_shape = (embedding_dim,))(x)

    model = Model(inputs=base_cnn.input, outputs=output, name=f"{backbone}_Embedding")
    return model

In [21]:
embedding_model = get_embedding_model(input_shape=(64,64,3), backbone='EfficientNetB0',layer_to_freeze='block7a')

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [73]:
len(embedding_model.non_trainable_weights), len(embedding_model.trainable_weights)

(300, 26)

In [ ]:
# for layer in embedding_model.layers:
#     print(f"{layer.name}: {layer.trainable}")

In [74]:
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.callbacks import ModelCheckpoint

lr_schedule = CosineDecay(
    initial_learning_rate=0.0001,   # Start warmup here
    decay_steps=1000,
    alpha=0.1,                      # Final LR = 0.1 * 0.001 = 0.0001
    warmup_target=0.001,           # Target LR after warmup
    warmup_steps=500
)

# ModelCheckpoint callback — saves the best model based on val_loss
checkpoint_callback = ModelCheckpoint(
    filepath="/kaggle/working/saved_models/EfficientNetB0_l2sp.keras",  # Must end with .keras for full model
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=False,
    mode="min",
    verbose=1
)

optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

embedding_model.compile(
    optimizer=optimizer,
    loss=TripletSemiHardLoss(margin=0.2, distance_metric='angular')
)

In [ ]:
# 10 + 10 + 30
history=embedding_model.fit(train_dataset,
                    steps_per_epoch=1000,
                    epochs=30,
                    validation_data=valid_dataset,
                    callbacks=[checkpoint_callback],
                    verbose=1)

## 8. Hyperparameter Tuning

In [23]:
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.callbacks import ModelCheckpoint
import tensorflow as tf
import keras_tuner
from loss.tripletloss import TripletSemiHardLoss


class TripletHyperModel(keras_tuner.HyperModel):
    def __init__(self, input_shape=(64, 64, 3), backbone='EfficientNetB0', layer_to_freeze='block7a'):
        self.input_shape = input_shape
        self.backbone = backbone
        self.layer_to_freeze = layer_to_freeze

    def build(self, hp):
        margin = hp.Float("margin", min_value=0.1, max_value=1.0, step=0.1)
        embedding_dim = hp.Choice("embedding_dim", values=[64, 128, 256, 512])
        distance_metric = hp.Choice("distance_metric",["L2", "squared-L2", "angular"])
        regularization_weight = hp.Float("regularization_weight", min_value=0.0001, max_value = 0.1)


        inputs = layers.Input(shape=self.input_shape)
        backbone_class = getattr(tf.keras.applications, self.backbone)
        base_cnn = backbone_class(include_top=False, input_tensor=inputs, weights="imagenet")

        freeze = False
        for layer in base_cnn.layers:
            if layer.name.startswith(self.layer_to_freeze):
                freeze = True
            layer.trainable = freeze
            
            if layer.trainable and hasattr(layer, 'kernel') and layer.get_weights():
                base_weight_tensor = layer.get_weights()[0]
                layer.kernel_regularizer = L2_SP(base_weights=base_weight_tensor, l2=regularization_weight)

        x = layers.Flatten()(base_cnn.output)
        x = layers.Dense(512, activation="relu",
                         kernel_regularizer=regularizers.l2(regularization_weight))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dense(embedding_dim, activation="relu",
                         kernel_regularizer=regularizers.l2(regularization_weight))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dense(embedding_dim,
                         kernel_regularizer=regularizers.l2(regularization_weight))(x)
        output = layers.Lambda(lambda x: tf.math.l2_normalize(x, axis=1), name="l2_normalized")(x)

        model = Model(inputs=inputs, outputs=output, name=f"{self.backbone}_Embedding")

        # Learning rate schedule
        lr_schedule = CosineDecay(
            initial_learning_rate=0.0001,
            decay_steps=1000,
            alpha=0.1,
            warmup_target=0.001,
            warmup_steps=500
        )

        optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

        model.compile(
            optimizer=optimizer,
            loss=TripletSemiHardLoss(margin=margin, distance_metric=distance_metric)
        )

        return model


In [ ]:
hypermodel = TripletHyperModel(input_shape=(64,64,3), layer_to_freeze='block7a')
# tuner = keras_tuner.RandomSearch(
#     hypermodel,
#     objective="val_loss",
#     max_trials=10,
#     directory="triplet_tuning",
#     project_name="effnet_margin"
# )

tuner = keras_tuner.Hyperband(
    hypermodel,
    objective="val_loss",
    max_epochs=10,
    factor=3,
    directory="triplet_tuning_v1",
    project_name="effnet_margin"
)

tuner.search(train_dataset, 
             validation_data=valid_dataset, 
             steps_per_epoch=1000, 
             epochs=10, 
             callbacks=[
                 ModelCheckpoint(
                    filepath="/kaggle/working/saved_models/EfficientNetB0_model.keras",
                    monitor="val_loss",
                    save_best_only=True,
                    save_weights_only=False,
                    mode="min",
                    verbose=1
                )
])


## 9. Evaluation of the trained model

### Create the test dataset

In [ ]:
!git clone https://github.com/notofonts/notofonts.github.io.git
!wget https://unifoundry.com/pub/unifont/unifont-16.0.04/font-builds/unifont-16.0.04.otf -O unifont-16.0.04.otf
!wget https://software.sil.org/downloads/r/charis/Charis-7.000.zip
!unzip Charis-7.000.zip -d .
!wget https://software.sil.org/downloads/r/gentium/Gentium-7.000.zip
!unzip Gentium-7.000.zip -d .
!wget https://software.sil.org/downloads/r/doulos/DoulosSIL-7.000.zip
!unzip DoulosSIL-7.000.zip -d .
!wget https://software.sil.org/downloads/r/andika/Andika-7.000.zip
!unzip Andika-7.000.zip -d .
!wget https://software.sil.org/downloads/r/galatia/GalatiaSIL-2.1-web.zip
!unzip GalatiaSIL-2.1-web.zip -d .
!wget https://catrinity-font.de/downloads/Catrinity.otf

!mkdir -p fonts
!mv unifont-16.0.04.otf fonts
!mv Charis-7.000/Charis-Regular.ttf fonts
!mv Gentium-7.000/Gentium-Regular.ttf fonts
!mv DoulosSIL-7.000/DoulosSIL-Regular.ttf fonts
!mv Andika-7.000/Andika-Regular.ttf fonts
!mv GalatiaSIL-2.1-web/GalSILB.ttf fonts
!mv GalatiaSIL-2.1-web/GalSILR.ttf fonts
!mv Catrinity.otf fonts

In [107]:
root_dirs = ["notofonts.github.io/fonts","fonts"]
font_paths = []
for root_dir in root_dirs:
  for root, dirs, files in os.walk(root_dir):
      for file in files:
          if file.endswith('.ttf') or file.endswith('.otf'):
              font_paths.append(os.path.join(root, file))
import os
from collections import defaultdict

# A map from font family name to its candidate font files
font_family_map = defaultdict(list)

def extract_family_name(path):
    filename = os.path.basename(path)
    # Remove style part like -Bold, -Thin, etc.
    name = filename.replace('.ttf', '')
    name = name.split('-')[0]  # e.g., NotoSansOriya-Regular → NotoSansOriya
    return name

# Group fonts by base family name
for path in font_paths:
    family = extract_family_name(path)
    font_family_map[family].append(path)

# Now pick the 'basic' font: prefer Regular.ttf, else first available
pruned_fonts = []

for family, paths in font_family_map.items():
    regular_fonts = [p for p in paths if 'Regular.ttf' in p]
    if regular_fonts:
        pruned_fonts.append(regular_fonts[0])
    else:
        pruned_fonts.append(paths[0])  # fallback

# Optional: sort the result
pruned_fonts.sort()

# Final output
# for font in pruned_fonts:
#     print(font)


In [ ]:
import os
import subprocess
from glob import glob

FONT_SOURCE_DIRS = ["notofonts.github.io/fonts","fonts"]
FONT_INSTALL_DIR = "/usr/share/fonts/truetype/custom/"

os.makedirs(FONT_INSTALL_DIR, exist_ok=True)

# Skip fonts with variable axes like [wght] or [wdth,wght]
def is_variable_font(font_path):
    basename = os.path.basename(font_path)
    return "[" in basename and "]" in basename

# Extensions to include
FONT_EXTENSIONS = ["ttf", "otf"]

# Loop through each font source directory
for src_dir in FONT_SOURCE_DIRS:
    for ext in FONT_EXTENSIONS:
        font_files = glob(os.path.join(src_dir, f"**/*.{ext}"), recursive=True)

        for font_file in font_files:
            if is_variable_font(font_file):
                print(f"Skipping variable font: {font_file}")
                continue

            filename = os.path.basename(font_file)
            installed_path = os.path.join(FONT_INSTALL_DIR, filename)
            subprocess.run(["cp", font_file, installed_path])

# Update font cache
subprocess.run(["fc-cache", "-fv"])

In [109]:
import subprocess

# Run fc-list and decode the output
output = subprocess.check_output(['fc-list'], encoding='utf-8')

# Split the output into lines (each line = one font)
font_list = output.strip().split('\n')
fc_font_names={}

for font in font_list:
  font_name = font.split(':')[1].strip().split(',')[0]
  font_name_striped = ''.join(font_name.split())
  if font_name_striped not in fc_font_names:
    fc_font_names[font_name_striped.lower()]=font_name

# fc_font_names

In [111]:
import os

# Acceptable general categories
renderable_categories = {
    'Lu', 'Ll', 'Lt', 'Lm', 'Lo',  # Letters
    'Mn', 'Mc', 'Me',              # Marks
    'Nd', 'Nl', 'No',              # Numbers
    'Pc', 'Pd', 'Ps', 'Pe', 'Pi', 'Pf', 'Po',  # Punctuation
    'Sm', 'Sc', 'Sk', 'So',        # Symbols
    # 'Zs'                           # Space separator
}

scripts_dir = 'scripts'
os.makedirs(scripts_dir, exist_ok=True)

for script, range_info in alphabets.items():
  start_codepoint = range_info['start']
  end_codepoint = range_info['end']

  start_int = int(start_codepoint, 16)
  end_int = int(end_codepoint, 16)

  with open('UnicodeData.txt' ,'r') as infile, open(os.path.join(scripts_dir,f'{script}.txt'),'w') as outfile:
    for line in infile.readlines():
      parts = line.split(';')

      if len(parts)<3:
        continue

      codepoint, name, category,_, _, decomposition = parts[0], parts[1], parts[2], parts[3], parts[4], parts[5]
      if decomposition and not decomposition.startswith('<'):
          continue
      cp_int = int(codepoint, 16)

      if start_int<=cp_int<=end_int:
        if category in renderable_categories:
          outfile.write('U+'+codepoint+'\n')

In [ ]:
import subprocess
from tqdm import tqdm

scripts_dir = 'scripts'
scripts = os.listdir(scripts_dir)
scripts.sort()

supported_fonts_mapping={}
def get_fonts_supporting_codepoint(hex_codepoint):
    cmd = ["fc-list", f":charset={hex_codepoint}"]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    fonts = result.stdout.strip().split('\n') if result.stdout else []
    cleaned = set()
    for line in fonts:
        parts = line.split(':')
        font_name = parts[1].strip().split(',')[0]
        cleaned.add(font_name)
    return cleaned

all_codepoints = []

for script in scripts:
  with open(os.path.join(scripts_dir, script), 'r') as infile:
    for line in infile:
      codepoint = line.strip().split('+')[1]
      all_codepoints.append(codepoint)

for codepoint in tqdm(all_codepoints, desc="Analyzing all codepoints"):
  font_names = get_fonts_supporting_codepoint(codepoint)
  if codepoint not in supported_fonts_mapping:
    supported_fonts_mapping['U+'+codepoint]=font_names
  else:
    supported_fonts_mapping['U+'+codepoint].update(font_names)


In [113]:
# List of fonts to exclude completely (case-insensitive match)
excluded_fonts = {"humor sans", "comic sans", "comic neue"}  # All lowercase

supported_fonts_mapping_trimmed = {}

for character, fonts in supported_fonts_mapping.items():
    # Exclude unwanted fonts using case-insensitive 'in'
    filtered_fonts = [
        font for font in fonts
        if all(excluded not in font.lower() for excluded in excluded_fonts)
    ]

    # Separate into Noto and others
    noto_fonts = [font for font in filtered_fonts if font.startswith('Noto')]
    other_fonts = [font for font in filtered_fonts if not font.startswith('Noto')]

    # Pick up to 2 Noto fonts
    selected_noto_fonts = sorted(noto_fonts)[:2]

    # Combine with other non-Noto fonts
    trimmed_fonts = selected_noto_fonts + other_fonts

    supported_fonts_mapping_trimmed[character] = sorted(trimmed_fonts)


In [ ]:
supported_fonts_mapping_trimmed['U+0061']

In [ ]:
import json

# Save to JSON file
json_filename = "supported_fonts_mapping_trimmed.json"

with open(json_filename, 'w', encoding='utf-8') as json_file:
    json.dump(supported_fonts_mapping_trimmed, json_file, indent=4, ensure_ascii=False)

print(f"Supported fonts mapping saved to {json_filename}")

In [124]:
!rm -rf supported_fonts_mapping_trimmed

In [ ]:
import json

# Set threshold
MAX_FONTS_PER_CODEPOINT = 5

# Trim to at most 5 fonts per codepoint
trimmed_supported_fonts_mapping = {
    cp: fonts[:MAX_FONTS_PER_CODEPOINT]
    for cp, fonts in supported_fonts_mapping_trimmed.items()
}

# Save to JSON file
json_filename = "supported_fonts_mapping_trimmed.json"

with open(json_filename, 'w', encoding='utf-8') as json_file:
    json.dump(trimmed_supported_fonts_mapping, json_file, indent=4, ensure_ascii=False)

print(f"Supported fonts mapping saved to {json_filename}")


In [ ]:
unsupported_chars = []

for script in scripts:
    script_name = script.split('.')[0]

    with open(os.path.join(scripts_dir, script), 'r', encoding='utf-8') as infile:
        for line in infile:
            line = line.strip()
            if not line:
                continue  # skip blank lines
            if line not in trimmed_supported_fonts_mapping or trimmed_supported_fonts_mapping[line] == []:
                unsupported_chars.append(line)

len(unsupported_chars)

In [ ]:
!apt-get -y update
!apt-get -y install libfreetype6 libcairo2 libsm6 libxext6 libfontconfig1 libxrender1 fontconfig libgl1-mesa-glx unzip
!wget https://gitlab.com/ldo/qahirah/-/archive/master/qahirah-master.tar.gz
!tar -xvzf qahirah-master.tar.gz
!mv qahirah-master qahirah
%cd qahirah
!pip install .
%cd ..

In [ ]:
!wget https://gitlab.com/ldo/python_freetype/-/archive/master/python_freetype-master.tar.gz
!tar -xvzf python_freetype-master.tar.gz
!mv python_freetype-master python_freetype
%cd python_freetype
!pip install .
%cd ..

In [ ]:
!gdown https://drive.google.com/uc?id=1asyDDtvGJ5eV2B_hcq1lMANqgFfxrdp8

In [ ]:
from source.vis_gen import VisualGenerator
from tqdm import tqdm

class CustomVisualGenerator(VisualGenerator):
  def generate_dataset_from_json_file(self, file_path, font_styles, antialiases):
    """
      Generates a dataset of rendered images from a JSON file mapping Unicode codepoints to font names.

      Args:
          file_path (str): Path to the JSON file containing a dictionary where keys are codepoints (e.g., "U+0041")
                          and values are lists of font names that support rendering that codepoint.
          font_styles (List[str]): A list of font style names (e.g., ["Regular", "Bold", "Italic"]) to use for rendering.
          antialiases (List[str]): A list of antialiasing options (e.g., ["Default", "None", "Grayscale"]) to apply
                                  during rendering.

      This method processes each codepoint and renders it using each combination of font name, style, and antialiasing
      setting provided. The rendered images are saved to the output directory specified in the class.

      Note:
          - Codepoints that cause errors during rendering are skipped with a warning.
          - This function assumes the fonts are already installed and accessible by name.
          - Output directory will be created if it doesn't exist.
    """
    out_dir_abs = self._get_out_dir_abs_and_check()

    with open(file_path, 'r') as f:
      trimmed_supported_fonts_mapping = json.load(f)

    print(f"Processing {len(trimmed_supported_fonts_mapping)} codepoints...")
    for codepoint, font_names in tqdm(trimmed_supported_fonts_mapping.items(), desc="Codepoints"):
      try:
        code_point = chr(int('0x' + codepoint[2:], 16))
        self.generate_dataset_from_list([code_point], font_names, font_styles, antialiases)
      except Exception as e:
        print(f"Error processing codepoint {codepoint}: {e}")
        continue

    self._check_out_dir = True

  def generate_dataset_from_list(self, code_points, font_names, font_styles, antialiases):
    # Check if out_dir exists and create if not
    out_dir_abs = self._get_out_dir_abs_and_check()

    for font_name in font_names:
      self.font_name = font_name
      for font_style in font_styles:
        self.font_style = font_style
        for antialias in antialiases:
          self.antialias = antialias
          self.visualize_list(code_points)

    # Flag flipped to False in self._get_out_dir_abs_and_check
    self._check_out_dir = True

  def visualize_list(self, code_points, x=None, y=None):
    # Check if out_dir exists and create if not
    out_dir_abs = self._get_out_dir_abs_and_check()

    # Visualize list of code points
    for idx, code_point in enumerate(code_points):
        self.visualize_single(code_point, False, x=x, y=y)

    # Flag flipped to False in self._get_out_dir_abs_and_check
    self._check_out_dir = True

In [128]:
!rm -rf data

In [ ]:
vg = CustomVisualGenerator(font_name='Noto Sans')

vg.image_size = 224
vg.font_size = 210
vg.out_dir = 'data'
mapping_file='supported_fonts_mapping_trimmed.json'

vg.generate_dataset_from_json_file(mapping_file, ['Regular'],['Default'])

In [ ]:
# Remove the files with the same codepoint
import os
from collections import defaultdict

# Path to your folder
folder_path = "data"

# Dictionary to track unique codepoints
unique_files = {}
all_files = os.listdir(folder_path)

# Loop through all files
for file_name in all_files:
    if not file_name.endswith(".png"):
        continue  # Skip non-png files
    codepoint = file_name.split("_")[0]
    if codepoint not in unique_files:
        unique_files[codepoint] = file_name
    else:
        os.remove(os.path.join(folder_path, file_name))  # Optional: delete duplicates

# List of unique files
final_file_list = list(unique_files.values())

# Print result
print(f"Total unique codepoints: {len(final_file_list)}")
print("Some of the kept files:")
print(final_file_list[:10])

In [101]:
import tensorflow as tf

IMG_SIZE = (64,64)

def preprocess_image(image_path, rescale=True):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    if rescale:
        image = image / 255.0
    return image

BATCH_SIZE = 64

test_image_paths = sorted(os.listdir('data'))

for i in range(len(test_image_paths)):
  test_image_paths[i]=os.path.join('data', test_image_paths[i])

test_dataset = tf.data.Dataset.from_tensor_slices(test_image_paths)

test_dataset = test_dataset.map(
    lambda x: preprocess_image(x, rescale=False),
    num_parallel_calls=tf.data.AUTOTUNE
)

test_dataset = test_dataset \
    .batch(BATCH_SIZE) \
    .prefetch(tf.data.AUTOTUNE)

### Generate embeddings

In [ ]:
import numpy as np
import tensorflow as tf

all_embeddings = []
for batch in test_dataset:
    batch_embeddings = embedding_model.predict(batch)
    all_embeddings.append(batch_embeddings)

all_embeddings = np.vstack(all_embeddings)

labels=[]
for img_file in sorted(os.listdir('data')):
  labels.append(img_file.split('_')[0])

labels_to_image_path={}
for img_file in sorted(os.listdir('data')):
  labels_to_image_path[img_file.split('_')[0]]=os.path.join('data', img_file)

embeddings = {
    label: embedding for label, embedding in zip(labels, all_embeddings)
}

# Map embeddings to image paths
embedding_to_image = {
    i: test_image_paths[i] for i in range(len(all_embeddings))
}

In [103]:
import cv2
# Calculate manhattan distance
def manhattan(emb1, emb2):
    dis = np.abs(emb1 - emb2)
    total_dis = np.sum(dis)
    return total_dis

def euclidean_distance(emb1, emb2):
    total_dis = np.linalg.norm(emb1 - emb2)
    return total_dis

def cosine_distance(emb1, emb2):
    # Assumes emb1 and emb2 are already L2-normalized
    similarity = np.dot(emb1, emb2)
    return 1 - similarity  # convert similarity to distance

def _sum_squared_distance_rgb(img1, img2):
    """Get normalized sum squared difference.

    Args:
        img1: np.ndarray, 3d array representing the first image with shape
            [image_height, image_width, 3]
        img2: np.ndarray, 3d array representing the second image

    Returns:
        distance: Float, sum square distance between two images
    """
    # Calculate sum squared distance
    distance = cv2.matchTemplate(img1, img2, cv2.TM_SQDIFF_NORMED)[0][0]
    return distance

In [104]:
import heapq

def get_clusters(selected_char,distance_metric, embeddings, n_candidates=10, distance_threshold=0.1):
    anchor_embedding = embeddings[selected_char]
    label_dis_pairs = []
    
    # Step 1: Compute distances to all other labels
    for label, emb in embeddings.items():
        if label != selected_char:
            distance = distance_metric(anchor_embedding, emb)
            label_dis_pairs.append((label, distance))
    
    
    # Step 2: Maintain top N 
    # n_candidates = 30
    top_n_heap = []
    
    for label, dis in label_dis_pairs:
        heapq.heappush(top_n_heap, (-dis, label))  # negative for max-heap
        if len(top_n_heap) > n_candidates:
            heapq.heappop(top_n_heap)
    
    # Convert back to positive distance and sort
    top_n = sorted([(-dis, label) for dis, label in top_n_heap])
    
    # Step 3: View results
    # print(f"Top N closest labels to {selected_char}:")
    # for dis, label in top_n:
    #     print(f"{label}: distance = {dis:.4f}")

    candidate_pool = set()
    candidate_pool = candidate_pool.union(
                set([entry[1] for entry in top_n]))

    confusables = []
    for candidate in candidate_pool:
        if selected_char == candidate:
            continue
    
        # labels_to_image_path
        img1 = cv2.imread(labels_to_image_path[selected_char])
        img2 = cv2.imread(labels_to_image_path[candidate])
    
        distance = _sum_squared_distance_rgb(img1, img2)
        if distance <= distance_threshold:
            confusables.append((candidate, distance))

    confusables.sort(key=lambda x: x[1])
    return confusables

In [105]:
import matplotlib.pyplot as plt
from PIL import Image

def plot_confusables(confusables, labels_to_image_path, max_cols=5, scale=4):
    """
    Plots confusable characters using their image paths and distance scores.

    Args:
        confusables: List of (codepoint, score) tuples.
        labels_to_image_path: Dict mapping Unicode codepoints (e.g., 'U+03B1') to image file paths.
        max_cols: Max number of columns in the plot grid.
        scale: Scaling factor for figure size per image.
    """
    num_items = len(confusables)
    num_cols = min(max_cols, num_items)
    num_rows = (num_items + num_cols - 1) // num_cols

    fig_width = num_cols * scale
    fig_height = num_rows * scale

    fig, axes = plt.subplots(num_rows, num_cols, figsize=(fig_width, fig_height), constrained_layout=True)
    axes = axes.flatten()

    for i, (codepoint, score) in enumerate(confusables):
        img_path = labels_to_image_path.get(codepoint)
        ax = axes[i]
        ax.axis('off')

        if img_path is None:
            print(f"Warning: No image found for {codepoint}")
            ax.text(0.5, 0.5, f'{codepoint}\nMissing', ha='center', va='center', fontsize=12)
            continue

        img = Image.open(img_path)
        ax.imshow(img, cmap='gray')
        ax.text(0.5, -0.15, f'{codepoint}\nScore: {score:.2f}', 
                ha='center', va='top', transform=ax.transAxes, fontsize=12)

    # Hide unused axes
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.show()


In [ ]:
plot_confusables(get_clusters('U+0041', distance_metric=cosine_distance, distance_threshold=0.2, embeddings=embeddings,n_candidates=10), labels_to_image_path)

In [ ]:
set(symmetric_confusables['U+004F'])

In [ ]:
from tqdm import tqdm

predicted_clusters={}
for selected_char in tqdm(codepoints):
  top_n = get_clusters(selected_char, distance_metric=euclidean_distance, distance_threshold=0.2, embeddings=embeddings)
  for codepoint, dist in top_n:
    if selected_char not in predicted_clusters:
      predicted_clusters[selected_char] = [codepoint]
    else:
      predicted_clusters[selected_char].append(codepoint)

In [109]:
def calculate_precision_and_recall(codepoints, consortium_confusables_dict, predicted_clusters):
    recall_list=[]
    precision_list=[]
    for selected_char in codepoints:
        if selected_char in consortium_confusables_dict and selected_char in predicted_clusters:
            actual = set(consortium_confusables_dict[selected_char])
            predicted = set(predicted_clusters[selected_char])
            if len(actual)==0 or len(predicted)==0:
                continue
            recall = len(actual.intersection(predicted))/len(actual)
            precision = len(actual.intersection(predicted))/len(predicted)
            recall_list.append(recall)
            precision_list.append(precision)
    mean_recall = sum(recall_list)/len(recall_list)
    mean_precision = sum(precision_list)/len(precision_list)

    return mean_recall, mean_precision

In [ ]:
calculate_precision_and_recall(codepoints, symmetric_confusables, predicted_clusters)